# A2 Replay Trace Review

Load replay results, compute aggregate metrics, review individual cases, and annotate pass/fail.

## 1. Load results

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

RESULTS_PATH = Path("../data/eval/replay_v1/results.jsonl")
ANNOTATIONS_PATH = Path("../data/eval/replay_v1/annotations.jsonl")

rows = []
with RESULTS_PATH.open() as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["correct"] = df["primary_label"] == df["true_class"]

print(f"Loaded {len(df)} results from {RESULTS_PATH}")
print(f"Dry run: {df['dry_run'].all()}")
print()
print("Status distribution:")
print(df["status"].value_counts().to_string())
print()
print("Class distribution (true):")
print(df["true_class"].value_counts().to_string())
print()
completed = df[df["status"] == "completed"]
print(f"Overall accuracy (completed only): {completed['correct'].mean():.4f} ({completed['correct'].sum()}/{len(completed)})")
print()
print("Severity distribution (completed):")
print(completed["severity"].value_counts().to_string())

## 2. Aggregate metrics

In [ ]:
completed = df[df["status"] == "completed"].copy()

print("=" * 60)
print("OVERALL ACCURACY")
print("=" * 60)
acc = completed["correct"].mean()
print(f"{acc:.4f} ({completed['correct'].sum()}/{len(completed)})")
print()

print("=" * 60)
print("PER-CLASS SENSITIVITY (recall)")
print("=" * 60)
classes = sorted(completed["true_class"].unique())
for cls in classes:
    subset = completed[completed["true_class"] == cls]
    sens = subset["correct"].mean()
    n = len(subset)
    marker = " <<<" if cls in ("Melanoma", "Basal_Cell_Carcinoma") else ""
    print(f"  {cls:30s}  {sens:.4f}  ({subset['correct'].sum()}/{n}){marker}")
print()

print("=" * 60)
print("AGREEMENT SCORE DISTRIBUTION")
print("=" * 60)
print(completed["agreement_score"].describe().to_string())
print()
print("Confidence tier counts:")
print(completed["confidence_tier"].value_counts().to_string())
print()

print("=" * 60)
print("SEVERITY DISTRIBUTION")
print("=" * 60)
print(completed["severity"].value_counts().to_string())
print()

print("=" * 60)
print("LATENCY (seconds)")
print("=" * 60)
elapsed = df["elapsed_s"]
print(f"  mean:   {elapsed.mean():.2f}")
print(f"  median: {elapsed.median():.2f}")
print(f"  max:    {elapsed.max():.2f}")
print(f"  min:    {elapsed.min():.2f}")
print()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

completed["severity"].value_counts().plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Severity distribution")
axes[0].set_ylabel("Count")

completed["agreement_score"].hist(ax=axes[1], bins=10, color="steelblue", edgecolor="white")
axes[1].set_title("Agreement score")
axes[1].set_xlabel("Score")

df["elapsed_s"].hist(ax=axes[2], bins=20, color="steelblue", edgecolor="white")
axes[2].set_title("Latency (s)")
axes[2].set_xlabel("Seconds")

plt.tight_layout()
plt.show()

## 3. Error analysis

In [ ]:
completed = df[df["status"] == "completed"].copy()
errors = completed[~completed["correct"]].copy()

print(f"Misclassifications: {len(errors)}/{len(completed)} ({len(errors)/len(completed)*100:.1f}%)")
print()

if len(errors) > 0:
    print("Confusion pairs (true -> predicted):")
    pairs = errors.groupby(["true_class", "primary_label"]).size().reset_index(name="count")
    pairs = pairs.sort_values("count", ascending=False)
    for _, row in pairs.iterrows():
        print(f"  {row['true_class']:30s} -> {row['primary_label']:30s}  n={row['count']}")
    print()

    display_cols = [
        "image_path", "true_class", "primary_label", "primary_confidence",
        "secondary_label", "secondary_confidence", "agreement_score",
        "severity", "confidence_level", "xai_focus", "xai_boundary",
    ]
    print("All misclassified cases:")
    display(errors[display_cols])
else:
    print("No misclassifications.")

## 4. Interactive single-image review

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display
from PIL import Image

annotations = []
# Load existing annotations if present
if ANNOTATIONS_PATH.exists():
    with ANNOTATIONS_PATH.open() as f:
        for line in f:
            line = line.strip()
            if line:
                annotations.append(json.loads(line))
    print(f"Loaded {len(annotations)} existing annotations")

annotated_paths = {a["image_path"] for a in annotations}

image_options = [(f"[{i}] {row['true_class']}/{Path(row['image_path']).name}", i) for i, row in df.iterrows()]

dropdown = widgets.Dropdown(options=image_options, description="Image:", layout=widgets.Layout(width="500px"))
output = widgets.Output()
notes_box = widgets.Textarea(description="Notes:", layout=widgets.Layout(width="500px", height="60px"))
pass_btn = widgets.Button(description="PASS", button_style="success", layout=widgets.Layout(width="100px"))
fail_btn = widgets.Button(description="FAIL", button_style="danger", layout=widgets.Layout(width="100px"))
save_btn = widgets.Button(description="Save annotations", button_style="info", layout=widgets.Layout(width="200px"))
status_label = widgets.Label(value=f"{len(annotations)}/{len(df)} annotated")


def show_image(change=None):
    idx = dropdown.value
    row = df.iloc[idx]
    with output:
        clear_output(wait=True)
        already = "[ANNOTATED] " if row["image_path"] in annotated_paths else ""
        print(f"{already}Index {idx}")
        print(f"{'='*60}")

        img_path = Path(row["image_path"])
        if img_path.exists():
            img = Image.open(img_path)
            img.thumbnail((300, 300))
            display(img)
        else:
            print(f"Image not found: {img_path}")

        print(f"True class:        {row['true_class']}")
        print(f"Predicted:         {row['primary_label']} ({row['primary_confidence']:.4f})")
        print(f"Secondary:         {row.get('secondary_label', 'N/A')} ({row.get('secondary_confidence', 0):.4f})")
        print(f"Agreement:         {row['agreement_score']:.2f} ({row.get('confidence_tier', 'N/A')})")
        print(f"Status:            {row['status']}")
        print(f"Severity:          {row.get('severity', 'N/A')}")
        print(f"Confidence level:  {row.get('confidence_level', 'N/A')}")
        print(f"Recapture needed:  {row.get('recapture_needed', 'N/A')}")
        print(f"Immed. referral:   {row.get('immediate_referral', 'N/A')}")
        print(f"Escalated:         {row.get('escalated', False)}")
        print(f"Advice allowed:    {row.get('treatment_advice_allowed', 'N/A')}")
        print()
        print(f"XAI focus:         {row.get('xai_focus', 'N/A')}")
        print(f"XAI boundary:      {row.get('xai_boundary', 'N/A')}")
        print(f"XAI artifact flag: {row.get('xai_artifact_flag', 'N/A')}")
        print()
        print(f"Summary: {row.get('summary_text', 'N/A')}")
        print(f"Action:  {row.get('recommended_action', 'N/A')}")
        print(f"Caveats: {row.get('confidence_caveats', [])}")
        print(f"Triage:  {row.get('triage_reasons', [])}")
        print(f"Warnings:{row.get('warnings', [])}")
        print(f"Messages:{row.get('messages', [])}")
        print()
        print(f"Patient: Fitz {row.get('patient_fitzpatrick', '?')}, age {row.get('patient_age', '?')}, spreading={row.get('patient_is_spreading', '?')}")
        print(f"Elapsed: {row.get('elapsed_s', 0):.2f}s")


def record_annotation(verdict):
    idx = dropdown.value
    row = df.iloc[idx]
    entry = {
        "image_path": row["image_path"],
        "true_class": row["true_class"],
        "primary_label": row.get("primary_label"),
        "verdict": verdict,
        "notes": notes_box.value.strip(),
    }
    # Replace if already annotated
    annotations[:] = [a for a in annotations if a["image_path"] != row["image_path"]] + [entry]
    annotated_paths.add(row["image_path"])
    notes_box.value = ""
    status_label.value = f"{len(annotations)}/{len(df)} annotated"
    # Advance to next
    current_options = [opt[1] for opt in image_options]
    pos = current_options.index(idx)
    if pos + 1 < len(current_options):
        dropdown.value = current_options[pos + 1]


def on_pass(btn):
    record_annotation("pass")


def on_fail(btn):
    record_annotation("fail")


def on_save(btn):
    ANNOTATIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with ANNOTATIONS_PATH.open("w") as f:
        for a in annotations:
            f.write(json.dumps(a) + "\n")
    status_label.value = f"Saved {len(annotations)} annotations to {ANNOTATIONS_PATH}"


dropdown.observe(show_image, names="value")
pass_btn.on_click(on_pass)
fail_btn.on_click(on_fail)
save_btn.on_click(on_save)

display(widgets.VBox([
    dropdown,
    output,
    notes_box,
    widgets.HBox([pass_btn, fail_btn, save_btn, status_label]),
]))

show_image()

## 5. Safety checks

In [ ]:
completed = df[df["status"] == "completed"].copy()

checks = [
    (
        "Melanoma predicted but NOT urgent",
        completed[(completed["primary_label"] == "Melanoma") & (completed["severity"] != "urgent")],
    ),
    (
        "BCC predicted with severity=routine",
        completed[(completed["primary_label"] == "Basal_Cell_Carcinoma") & (completed["severity"] == "routine")],
    ),
    (
        "Tinea + spreading rapidly but NOT urgent",
        completed[
            (completed["primary_label"] == "Tinea")
            & (completed["patient_is_spreading"] == True)
            & (completed["severity"] != "urgent")
        ],
    ),
    (
        "immediate_referral=True but severity != urgent",
        completed[
            (completed["immediate_referral"] == True)
            & (completed["severity"] != "urgent")
        ],
    ),
]

any_found = False
for name, subset in checks:
    if len(subset) > 0:
        any_found = True
        print(f"WARNING: {name} ({len(subset)} cases)")
        display(subset[["image_path", "true_class", "primary_label", "primary_confidence",
                        "severity", "immediate_referral", "patient_is_spreading", "triage_reasons"]])
        print()
    else:
        print(f"OK: {name} — none found")

if not any_found:
    print("\nAll safety checks passed.")

print()
print("=" * 60)
print("WARNINGS AND MESSAGES SUMMARY")
print("=" * 60)
has_warnings = df[df["warnings"].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)]
has_messages = df[df["messages"].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)]
print(f"Images with warnings: {len(has_warnings)}")
print(f"Images with messages: {len(has_messages)}")
if len(has_warnings) > 0:
    print("\nWarnings:")
    for _, row in has_warnings.iterrows():
        print(f"  {Path(row['image_path']).name}: {row['warnings']}")
if len(has_messages) > 0:
    print("\nMessages:")
    for _, row in has_messages.iterrows():
        print(f"  {Path(row['image_path']).name}: {row['messages']}")

## 6. Save summary

In [ ]:
print(f"Annotation progress: {len(annotations)}/{len(df)} reviewed")
if annotations:
    pass_count = sum(1 for a in annotations if a["verdict"] == "pass")
    fail_count = sum(1 for a in annotations if a["verdict"] == "fail")
    print(f"  Pass: {pass_count}")
    print(f"  Fail: {fail_count}")
    if fail_count > 0:
        print("\nFailed cases:")
        for a in annotations:
            if a["verdict"] == "fail":
                print(f"  {Path(a['image_path']).name}: {a['true_class']} -> {a.get('primary_label', '?')}  notes: {a.get('notes', '')}")

# Auto-save on final cell
if annotations:
    ANNOTATIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with ANNOTATIONS_PATH.open("w") as f:
        for a in annotations:
            f.write(json.dumps(a) + "\n")
    print(f"\nAnnotations saved to {ANNOTATIONS_PATH}")
else:
    print("\nNo annotations yet — use the interactive review in cell 4.")